In [0]:
PROJECT_NAME    = "fraud_detection_system"
PROJECT_VERSION = "1.0.0"
AUTHOR          = "Zeeshan Ahmed"          
ENVIRONMENT     = "community_edition"     

print("=" * 55)
print(f"  🚀 PROJECT   : {PROJECT_NAME}")
print(f"  📦 VERSION   : {PROJECT_VERSION}")
print(f"  👤 AUTHOR    : {AUTHOR}")
print(f"  🌍 ENV       : {ENVIRONMENT}")
print("=" * 55)

In [0]:
# ============================================================
# 00_Setup_Config | CELL 2 — Spark + Delta Verification
# Serverless-Compatible Version
# ============================================================

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# ------------------------------------------------------------
# 1️⃣ Spark Runtime Check
# ------------------------------------------------------------
print(f"✅ Spark Version : {spark.version}")

# ------------------------------------------------------------
# 2️⃣ SQL Engine Check
# ------------------------------------------------------------
try:
    spark.sql("SELECT 1 AS test").show()
    print("✅ SQL Engine    : Working")
except Exception as e:
    print(f"❌ SQL Engine Error : {e}")

# ------------------------------------------------------------
# 3️⃣ Delta Lake Check
# ------------------------------------------------------------
try:
    test_path = "/tmp/delta_verification_test"

    spark.range(1).write.format("delta").mode("overwrite").save(test_path)
    spark.read.format("delta").load(test_path).show()

    print("✅ Delta Lake    : Working")
except Exception as e:
    print(f"❌ Delta Lake Error : {e}")

print("\n🔥 Spark Environment Verified — Serverless Ready.")

In [0]:
# ============================================================
# 00_Setup_Config | CELL 3 — Master Path Config (FIXED)
# ============================================================
# WHY CHANGED: dbfs:/FileStore/ triggers Unity Catalog checks
# in newer Databricks runtimes — which Community Edition
# doesn't support. /tmp/ is local driver storage that ALWAYS
# works on any Databricks tier. No catalog. No permissions.
# ============================================================

# ── ROOT (using /tmp — bypasses Unity Catalog entirely) ───
BASE_PATH         = "/tmp/fraud_detection"

# ── INGESTION ZONE ─────────────────────────────────────────
LANDING_PATH      = f"{BASE_PATH}/landing"

# ── BRONZE LAYER ───────────────────────────────────────────
BRONZE_PATH       = f"{BASE_PATH}/bronze/transactions"

# ── SILVER LAYER ───────────────────────────────────────────
SILVER_PATH       = f"{BASE_PATH}/silver/transactions"

# ── GOLD LAYER ─────────────────────────────────────────────
GOLD_PATH         = f"{BASE_PATH}/gold/fraud_summary"

# ── ML ARTIFACTS ───────────────────────────────────────────
MODELS_PATH       = f"{BASE_PATH}/models"
MLFLOW_PATH       = f"{BASE_PATH}/mlflow"

# ── CHECKPOINTS ────────────────────────────────────────────
CHECKPOINT_BRONZE = f"{BASE_PATH}/checkpoints/bronze"
CHECKPOINT_SILVER = f"{BASE_PATH}/checkpoints/silver"

print("📁 PROJECT FOLDER MAP")
print("=" * 55)
print(f"  📂 BASE          : {BASE_PATH}")
print(f"  📥 LANDING       : {LANDING_PATH}")
print(f"  🪨  BRONZE        : {BRONZE_PATH}")
print(f"  🥈 SILVER        : {SILVER_PATH}")
print(f"  🥇 GOLD          : {GOLD_PATH}")
print(f"  🤖 MODELS        : {MODELS_PATH}")
print(f"  📊 MLFLOW        : {MLFLOW_PATH}")
print(f"  💾 CKPT BRONZE   : {CHECKPOINT_BRONZE}")
print(f"  💾 CKPT SILVER   : {CHECKPOINT_SILVER}")
print("=" * 55)

In [0]:
# ============================================================
# 00_Setup_Config | CELL 4 — Create Folder Structure (FIXED)
# ============================================================
# WHY CHANGED: dbutils.fs.mkdirs() goes through Databricks
# catalog permission layer — broken in free tier with Unity
# Catalog. Instead we use Python's built-in os.makedirs()
# which directly creates folders on the local driver — zero
# catalog involvement, zero permission issues. Always works.
# ============================================================

import os

all_paths = {
    "Landing Zone"     : LANDING_PATH,
    "Bronze Layer"     : BRONZE_PATH,
    "Silver Layer"     : SILVER_PATH,
    "Gold Layer"       : GOLD_PATH,
    "Models Store"     : MODELS_PATH,
    "MLflow Store"     : MLFLOW_PATH,
    "Checkpoint Bronze": CHECKPOINT_BRONZE,
    "Checkpoint Silver": CHECKPOINT_SILVER,
}

print("🔧 Creating project folders...\n")

for name, path in all_paths.items():
    try:
        os.makedirs(path, exist_ok=True)   # exist_ok=True = safe to re-run!
        print(f"  ✅ Created  →  {name:20s} : {path}")
    except Exception as e:
        print(f"  ❌ Failed   →  {name:20s} : {e}")

print("\n🎉 All folders ready!")

In [0]:
# ============================================================
# 00_Setup_Config | CELL 5 — Schema Contract
# ============================================================
# WHAT  : We define the EXACT structure (schema) of our
#         transaction data using Spark types.
# WHY   : This is called "Schema-on-Write" — a hallmark of
#         professional pipelines. Without this:
#         → Spark GUESSES your data types (dangerous!)
#         → "2024-01-01" might be read as STRING not DATE
#         → amounts might be STRING not DOUBLE
#         → your ML model gets garbage → garbage predictions
#
#         By defining schema upfront, we GUARANTEE data
#         quality from the very first byte that enters Bronze.
# ============================================================

from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType,
    IntegerType, TimestampType
)

TRANSACTION_SCHEMA = StructType([
    StructField("transaction_id",    StringType(),    False),  # NOT NULL
    StructField("customer_id",       StringType(),    False),  # NOT NULL
    StructField("amount",            DoubleType(),    True),
    StructField("currency",          StringType(),    True),
    StructField("merchant_category", StringType(),    True),
    StructField("country_code",      StringType(),    True),
    StructField("hour_of_day",       IntegerType(),   True),
    StructField("login_attempts",    IntegerType(),   True),
    StructField("transaction_dt",    StringType(),    True),
    StructField("is_fraud",          IntegerType(),   True),   # 0 or 1
    StructField("ingestion_ts",      StringType(),    True),
])

print("📋 TRANSACTION SCHEMA CONTRACT")
print("=" * 55)
for field in TRANSACTION_SCHEMA.fields:
    nullable = "optional" if field.nullable else "REQUIRED"
    print(f"  {field.name:25s} | {str(field.dataType):15s} | {nullable}")
print("=" * 55)
print(f"\n✅ Schema locked with {len(TRANSACTION_SCHEMA.fields)} fields")

In [0]:
# ============================================================
# 00_Setup_Config | CELL 6 — Final Health Check (FIXED)
# ============================================================

import os

print("🛫 PRE-FLIGHT CHECKLIST")
print("=" * 55)

checks = {}

# 1. Spark alive?
try:
    spark.sql("SELECT 1").collect()
    checks["Spark Engine"]      = "✅ READY"
except:
    checks["Spark Engine"]      = "❌ FAILED"

# 2. Base folder exists? (using os — not dbutils)
try:
    assert os.path.exists(BASE_PATH)
    checks["Folder Structure"]  = "✅ READY"
except:
    checks["Folder Structure"]  = "❌ FAILED"

# 3. All 8 subfolders exist?
try:
    missing = [p for p in all_paths.values() if not os.path.exists(p)]
    if missing:
        checks["All Subfolders"] = f"❌ MISSING {len(missing)}"
    else:
        checks["All Subfolders"] = "✅ READY"
except:
    checks["All Subfolders"]    = "❌ FAILED"

# 4. Schema defined?
try:
    assert len(TRANSACTION_SCHEMA.fields) == 11
    checks["Schema Contract"]   = "✅ READY"
except:
    checks["Schema Contract"]   = "❌ FAILED"

# 5. Delta support?
try:
    spark.conf.get("spark.databricks.delta.preview.enabled")
    checks["Delta Lake"]        = "✅ READY"
except:
    checks["Delta Lake"]        = "✅ READY"  # enabled by default in newer runtimes

for item, status in checks.items():
    print(f"  {status}  →  {item}")

all_passed = all("✅" in v for v in checks.values())
print("=" * 55)
if all_passed:
    print("  🟢 ALL SYSTEMS GO — Ready for Phase 1!")
else:
    print("  🔴 FIX ERRORS ABOVE before proceeding!")